In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from src.results_paper import *


In [ ]:
savepath = "../results_paper_SOP/"
create_folder(savepath)

## <b> FIGURE 1 </b>

$P_0 = 100$ (seeds)  
$p_0 = 1.0$ (initial probability)  
Network $1000$ x $1000$ -> $L$ x $N_{samples}$  
$L = 1000$  
$N_{samples} = 1000$  
$k = 10^{-5}$  
$N_T = 200$


In [ ]:
def plot_bond_network(filepath, savepath=None, dpi=600, min_density=1):
    """
    Plots only the bond connections (black segments) between active sites in the lattice.
    The image is vertically cropped to include only rows with density ≥ min_density.
    The y-axis is inverted (matching the orientation in the original paper).
    """
    data = np.load(filepath)
    network = data["network"]
    nrows, ncols = network.shape

    segments = []
    density_per_row = np.zeros(nrows, dtype=int)

    for i in range(nrows):
        for j in range(ncols):
            if network[i, j] == 1:
                # right bond
                if j + 1 < ncols and network[i, j + 1] == 1:
                    segments.append([(j, i), (j + 1, i)])
                    density_per_row[i] += 1
                # upward bond
                if i + 1 < nrows and network[i + 1, j] == 1:
                    segments.append([(j, i), (j, i + 1)])
                    density_per_row[i] += 1

    active_rows = np.where(density_per_row >= min_density)[0]
    if active_rows.size == 0:
        print("[!] No row found with minimum density.")
        return
    row_start = active_rows[0]
    row_end = active_rows[-1]

    # Filter visible bonds
    segments_filtered = [
        [(x0, y0), (x1, y1)]
        for (x0, y0), (x1, y1) in segments
        if row_start <= y0 <= row_end and row_start <= y1 <= row_end
    ]

    # Plotting
    fig, ax = plt.subplots(figsize=(8, 8))
    lc = LineCollection(segments_filtered, colors="black", linewidths=0.25)
    ax.add_collection(lc)
    ax.set_xlim(0, ncols)
    ax.set_ylim(row_end, row_start + 1)
    ax.invert_yaxis()  # ensure top-to-bottom growth
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout(pad=0)

    if savepath:
        plt.savefig(savepath, dpi=dpi, bbox_inches="tight", pad_inches=0)
        print(f"[✔] Image saved to: {savepath}")
    else:
        plt.show()

In [ ]:
path = "../Data/bond_percolation/dim_2/L_1000_N_samples_1000/NT_constant/NT_200/k_1.0e-05/network/"
filename = list_npz_files()[0]
plot_bond_network(path + filename, savepath=savepath + "Fig1.pdf", dpi=600, min_density=1)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Caminho onde os arquivos estão
path = "../Data/bond_percolation/dim_2/L_1000_N_samples_1000/NT_constant/NT_200/k_1.0e-05/network/"

# Lista de arquivos .npz no diretório
npz_files = [f for f in os.listdir(path) if f.endswith(".npz")]

# Lista para armazenar o último tempo (linha t) com valor 1
last_time_with_1 = []

for filename in npz_files:
    data = np.load(os.path.join(path, filename))
    network = data["network"]  # shape: (N_samples, L)

    # Percorre de trás para frente, procurando a última linha com pelo menos um 1 positivo
    for t in range(network.shape[0] - 1, -1, -1):
        if np.any(network[t, :] == 1):  # valor 1 positivo
            last_time_with_1.append(t)
            break
    else:
        last_time_with_1.append(-1)  # opcional: se não houver nenhum 1

# Remover -1 se não quiser contar arquivos sem valor 1
last_time_with_1 = [t for t in last_time_with_1 if t != -1]

# Plot do histograma: número de arquivos (redes) que percolaram até o tempo t
plt.figure(figsize=(8, 6))
plt.hist(last_time_with_1, bins=np.arange(min(last_time_with_1), max(last_time_with_1)+2)-0.5, edgecolor='black', align='mid')
plt.title("Último tempo $t$ com valor 1 positivo (por rede)")
plt.xlabel("Tempo máximo $t$ com valor 1")
plt.ylabel("Número de redes")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
path = "../animation/"
filename = "P0_0.10_p0_1.00_seed_630136907.npz"
data = np.load(path + filename)
network = data["network"]

max_value = np.max(network)
indices = np.argwhere(network == max_value)

print(f"Maior valor: {max_value}")
print(f"Total de ocorrências: {len(indices)}")
print("Índices:")
for idx in indices:
    print(tuple(idx))


In [ ]:
path = "../animation/"
filename = "P0_0.10_p0_1.00_seed_630136907.npz"
data = np.load(path + filename)
network = data["network"]
print(network)

## <b> FIGURE 2 </b>

Network $2000$ x $2000$ -> $L$ x $N_{samples}$  
$L = 2000$  
$N_{samples} = 2000$  
$k = 10^{-4}$  
$N_T = 200$  
$P_0 = 100$ (seeds)  

(a)  
$p_0 = [0.4, 0.8, 1.0]$ (initial probability)  

(b)  
$p_0 = [0.3, 0.7, 1.0]$ (initial probability)  

In [ ]:
L = 2000
N_samples = 2000
k = 1.0e-04
Nt = 200
dim = 2
prop = "Pt"
type_Nt = 0

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(8, 8), sharex=True)


colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
fontsize_ticks = 13
thickness = 1.4
# --- Node Percolation ---
type_percolation_node = "node"
prop_node_names = list_npy_files(type_percolation_node, dim, L, N_samples, type_Nt, Nt, k, prop)
p_0_node = [0.4, 0.8, 1.0]

count = 0

for prop_ in prop_node_names:
    p_0 = os.path.basename(prop_)[11:14]
    if(float(p_0) in p_0_node):
        t, pt = load_t_pt(prop_)
        p_m = np.mean(pt[200:])
        
        axs[0].plot(t, pt, label=f"$p_0 = {p_0}$, $\\langle p \\rangle =${p_m:.4f}",color=colors[count])
        count += 1

axs[0].set_title("Node Percolation")
axs[0].legend(framealpha=0.0,fontsize=fontsize_ticks)
axs[0].set_xlim([0, 1000])
axs[0].set_ylabel("$p(t)$",fontsize=fontsize_ticks)
axs[0].tick_params(axis='both', labelsize=fontsize_ticks, width=thickness, length = 4.0, direction='in')
# Define a espessura das bordas
for spine in axs[0].spines.values():
    spine.set_linewidth(thickness)

# --- Bond Percolation ---
type_percolation_bond = "bond"
prop_bond_names = list_npy_files(type_percolation_bond, dim, L, N_samples, type_Nt, Nt, k, prop)
p_0_bond = [0.3, 0.7, 1.0]
count = 0
for prop_ in prop_bond_names:
    p_0 = os.path.basename(prop_)[11:14]
    if(float(p_0) in p_0_bond):
        t, pt = load_t_pt(prop_)
        p_m = np.mean(pt[200:])
        
        axs[1].plot(t, pt, label=f"$p_0 = {p_0}$, $\\langle p \\rangle =${p_m:.4f}", color=colors[count])
        count += 1

axs[1].set_title("Bond Percolation")
axs[1].legend(framealpha=0.0,fontsize=fontsize_ticks)
axs[1].set_xlim([0, 1000])
axs[1].set_xlabel("time",fontsize=fontsize_ticks)
axs[1].set_ylabel("$p(t)$",fontsize=fontsize_ticks)
axs[1].tick_params(axis='both', labelsize=fontsize_ticks, width=thickness, length = 4.0, direction='in')
# Define a espessura das bordas
for spine in axs[1].spines.values():
    spine.set_linewidth(thickness)

plt.tight_layout()
plt.show()
